# AI-Based Smart Waste Segregation and Classification System
### Course Project — Model Training & Evaluation Notebook

**Team Members:**
- Ruthikraj Nayak (2024BVS0079)
- Sai Kiran (2024BCS0087)
- Lakshmi Narayana (2024BEC0047)
- Santosh Kumar (2024BCS0175)

**Model Architecture:** MobileNetV2 with Transfer Learning & Fine-Tuning  
**Target Classes (5):** Glass, Paper, Metal, Organic Waste, Plastic  
**Dataset:** Real-World Waste Classification Dataset - 5 Class (Kaggle)

---
### Notebook Workflow
1. **Environment Setup & GPU Check**
2. **Dataset Acquisition & Preprocessing** (Organizing Turkish folder names into English classes: Cam Atik -> Glass, Kagit Atik -> Paper, Metal Atik -> Metal, Organik Atik -> Organic Waste, Plastik Atik -> Plastic)
3. **Train / Validation / Test Splitting** (70% Train, 15% Validation, 15% Test — Zero Data Leakage)
4. **Data Augmentation & MobileNetV2 Normalization**
5. **Transfer Learning Model Construction**
6. **Stage 1: Training Classification Head**
7. **Stage 2: Fine-Tuning Top Layers**
8. **Evaluation on Test Set (Accuracy, Loss, Classification Report, Confusion Matrix)**
9. **Exporting Model (`waste_mobilenetv2.keras`, `class_indices.json`) for Flask Deployment**

## 1. Environment Setup & GPU Verification
Colab provides free GPUs (T4). Enable GPU via: `Runtime` -> `Change runtime type` -> `T4 GPU`.

In [ ]:
import os
import shutil
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU Detected: {gpus[0].name}")
else:
    print("Running on CPU. For faster training, enable GPU under Runtime -> Change runtime type.")

# Set seeds for reproducible results
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

## 2. Dataset Download & Extraction
Choose either **Option A** (upload your `archive.zip` file directly to Colab) or **Option B** (use Kaggle API).

In [ ]:
# OPTION A: If you upload the zip file from your computer to Colab:
# (Upload your dataset zip file in the files panel on the left, then run this)
import zipfile

zip_filename = "archive (1).zip"  # or whatever name your zip file has
if os.path.exists(zip_filename):
    print(f"Extracting {zip_filename}...")
    with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
        zip_ref.extractall("dataset_raw")
    print("Extraction complete!")
else:
    print(f"{zip_filename} not found. Please upload your dataset zip file or specify its path.")

In [ ]:
# OPTION B: Download directly using Kaggle API (Alternative)
# !pip install -q kaggle
# from google.colab import files
# files.upload() # Upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d tuhanasinan/real-world-waste-classification-dataset-5-class
# !unzip -q real-world-waste-classification-dataset-5-class.zip -d dataset_raw

## 3. Dataset Inspection & Folder Mapping
The dataset folder contains Turkish category names. We map them cleanly to English presentation labels:
- `Cam Atik` -> **Glass**
- `Kagit Atik` -> **Paper**
- `Metal Atik` -> **Metal**
- `Organik Atik` -> **Organic Waste**
- `Plastik Atik` -> **Plastic**

In [ ]:
# Locate the root folder containing the 5 subfolders
raw_search_root = "dataset_raw"
dataset_root = None
for root, dirs, files in os.walk(raw_search_root):
    if any("cam" in d.lower() for d in dirs) and any("plastik" in d.lower() for d in dirs):
        dataset_root = root
        break

if dataset_root is None:
    print("Could not automatically locate the dataset folder. Please set dataset_root manually.")
else:
    print(f"Found dataset root: {dataset_root}")
    print("Subdirectories:", os.listdir(dataset_root))

FOLDER_MAPPING = {
    "Cam Atik": "Glass",
    "Kagit Atik": "Paper",
    "Metal Atik": "Metal",
    "Organik Atik": "Organic Waste",
    "Plastik Atik": "Plastic"
}

## 4. Train / Validation / Test Splitting (Zero Data Leakage)
We split the images physically into `train` (70%), `val` (15%), and `test` (15%) directories.
The **test** set remains completely untouched until final evaluation.

In [ ]:
output_split_dir = "waste_dataset_split"
splits = ["train", "val", "test"]
classes = list(FOLDER_MAPPING.values())

# Create destination folders
for s in splits:
    for c in classes:
        os.makedirs(os.path.join(output_split_dir, s, c), exist_ok=True)

# Copy and split images
for raw_folder, eng_label in FOLDER_MAPPING.items():
    # Match folder loosely for unicode flexibility
    matched = None
    for d in os.listdir(dataset_root):
        if raw_folder.split()[0].lower() in d.lower():
            matched = d
            break
    
    src_dir = os.path.join(dataset_root, matched)
    imgs = [f for f in os.listdir(src_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    np.random.shuffle(imgs)
    
    n_total = len(imgs)
    n_train = int(0.70 * n_total)
    n_val = int(0.15 * n_total)
    
    train_imgs = imgs[:n_train]
    val_imgs = imgs[n_train:n_train + n_val]
    test_imgs = imgs[n_train + n_val:]
    
    print(f"{eng_label:14s} | Total: {n_total:4d} | Train: {len(train_imgs):4d} | Val: {len(val_imgs):3d} | Test: {len(test_imgs):3d}")
    
    for f in train_imgs:
        shutil.copy(os.path.join(src_dir, f), os.path.join(output_split_dir, "train", eng_label, f))
    for f in val_imgs:
        shutil.copy(os.path.join(src_dir, f), os.path.join(output_split_dir, "val", eng_label, f))
    for f in test_imgs:
        shutil.copy(os.path.join(src_dir, f), os.path.join(output_split_dir, "test", eng_label, f))

print("\nAll images categorized and split successfully!")

## 5. Build tf.data Pipelines & MobileNetV2 Preprocessing
MobileNetV2 expects input images scaled to the range `[-1.0, 1.0]` using `preprocess_input`.
Data augmentation (rotation, flip, zoom) is applied strictly to the training set.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(output_split_dir, "train"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(output_split_dir, "val"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(output_split_dir, "test"),
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="categorical",
    shuffle=False
)

class_names = train_ds.class_names
print("Class labels (in alphabetical order):", class_names)

# Export class_indices.json for the Flask application
class_indices = {str(i): name for i, name in enumerate(class_names)}
with open("class_indices.json", "w", encoding="utf-8") as f:
    json.dump(class_indices, f, indent=2)
print("Saved class_indices.json:", class_indices)

# Data augmentation pipeline
augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1)
], name="data_augmentation")

# MobileNetV2 normalization
preprocess_layer = layers.Lambda(preprocess_input, name="mobilenetv2_preprocess")

train_ds = train_ds.map(lambda x, y: (preprocess_layer(augmentation(x, training=True)), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(lambda x, y: (preprocess_layer(x), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(lambda x, y: (preprocess_layer(x), y), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

## 6. Build MobileNetV2 Model Architecture
- **Base Model**: MobileNetV2 pre-trained on ImageNet (lightweight, highly efficient for mobile/web deployment).
- **Feature Extraction**: Base weights initially frozen.
- **Custom Classification Head**: Global Average Pooling -> Batch Normalization -> Dense(128, ReLU) -> Dropout(0.35) -> Dense(5, Softmax).

In [ ]:
base_model = MobileNetV2(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False  # Freeze base model

inputs = tf.keras.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D(name="gap")(x)
x = layers.BatchNormalization(name="bn")(x)
x = layers.Dense(128, activation="relu", name="dense_128")(x)
x = layers.Dropout(0.35, name="dropout")(x)
outputs = layers.Dense(len(class_names), activation="softmax", name="predictions")(x)

model = models.Model(inputs=inputs, outputs=outputs, name="SmartWaste_MobileNetV2")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 7. Stage 1: Train Classification Head (Feature Extraction)
Train only the newly attached dense layers while the pre-trained feature extractor is frozen.

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-6),
    tf.keras.callbacks.ModelCheckpoint("best_head_model.keras", monitor="val_loss", save_best_only=True)
]

history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)

## 8. Stage 2: Fine-Tuning Top Layers of MobileNetV2
We unfreeze the top layers of MobileNetV2 and train with a very small learning rate (`1e-5`) so that the model adapts to subtle textures and reflections of waste materials (glass shine, plastic transparency, organic food textures).

In [ ]:
base_model.trainable = True
# Keep the earlier 100 low-level edge/texture layers frozen; fine-tune the high-level semantic layers
for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=25,
    initial_epoch=len(history_stage1.epoch),
    callbacks=callbacks
)

## 9. Rigorous Evaluation on the Unseen Test Set
Evaluate actual generalization capability on test images that the model has never encountered before.

In [ ]:
print("Evaluating model on test dataset...")
test_loss, test_acc = model.evaluate(test_ds)
print(f"\nTest Accuracy: {test_acc * 100:.2f}%")
print(f"Test Loss:     {test_loss:.4f}")

# Collect predictions
y_true_list = []
y_pred_list = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true_list.extend(np.argmax(labels.numpy(), axis=1))
    y_pred_list.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true_list)
y_pred = np.array(y_pred_list)

print("\n--- Detailed Classification Report ---")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# Confusion Matrix Visualization
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix — Test Set Evaluation", fontsize=14, fontweight="bold", pad=12)
plt.xlabel("Predicted Category", fontsize=12)
plt.ylabel("Actual True Category", fontsize=12)
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=300)
plt.show()

## 10. Training History Curves
Plotting loss and accuracy over epochs for presentation documentation.

In [ ]:
# Combine training metrics across both stages
acc = history_stage1.history['accuracy'] + history_stage2.history['accuracy']
val_acc = history_stage1.history['val_accuracy'] + history_stage2.history['val_accuracy']
loss = history_stage1.history['loss'] + history_stage2.history['loss']
val_loss = history_stage1.history['val_loss'] + history_stage2.history['val_loss']

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy', color='#16a34a', linewidth=2)
plt.plot(val_acc, label='Validation Accuracy', color='#0284c7', linewidth=2, linestyle='--')
plt.axvline(x=len(history_stage1.epoch)-1, color='red', linestyle=':', label='Fine-Tuning Start')
plt.title('Model Accuracy vs Epochs', fontsize=13, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss', color='#dc2626', linewidth=2)
plt.plot(val_loss, label='Validation Loss', color='#ea580c', linewidth=2, linestyle='--')
plt.axvline(x=len(history_stage1.epoch)-1, color='red', linestyle=':', label='Fine-Tuning Start')
plt.title('Model Loss vs Epochs', fontsize=13, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=300)
plt.show()

## 11. Save & Download Model Files for Flask
Save the model in `.keras` and `.tflite` formats, and download them along with `class_indices.json` directly to your computer.
After downloading, move `waste_mobilenetv2.keras` and `class_indices.json` into your project's `model/` folder!

In [ ]:
# Save modern Keras format
model.save("waste_mobilenetv2.keras")
print("Saved waste_mobilenetv2.keras")

# Export lightweight TFLite model
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open("waste_mobilenetv2.tflite", "wb") as f:
    f.write(tflite_model)
print("Saved waste_mobilenetv2.tflite")

# Automatic Colab download helper
try:
    from google.colab import files
    print("Downloading model files to your computer...")
    files.download("waste_mobilenetv2.keras")
    files.download("class_indices.json")
    files.download("confusion_matrix.png")
    files.download("training_curves.png")
except Exception as e:
    print("To download manually, right click on the file in Colab's left files panel -> Download.")